# Notebook 08 — Reviewer Segmentation

**Project:** Starbucks Customer Voice Intelligence: A U.S. Coffee Chain Market Study  
**Purpose:** Examine how reviewer experience level and review length relate to rating outcomes — identifying whether Starbucks's negative signal is concentrated among specific reviewer profiles.

## 0. Environment setup

In [1]:

import sys
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

INTERIM_DIR = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Environment ready.")


Environment ready.


## 1. Load data

In [2]:
df   = pd.read_parquet(INTERIM_DIR / "analysis_ready.parquet")
sbux = df[df["brand_category"] == "Starbucks"]

print(f"Starbucks reviews: {len(sbux):,}")
print(f"\nReviewer tiers:")
print(sbux["user_activity_tier"].value_counts().to_string())

Starbucks reviews: 11,675

Reviewer tiers:
user_activity_tier
Casual     3778
Regular    3385
Power      2468
Elite      2044


## 2. Rating profile by reviewer tier

Reviewers are segmented into four tiers based on total Yelp review activity: **Casual** (fewest reviews), **Regular**, **Power**, and **Elite** (most prolific). The question is whether the source of Starbucks's negative signal is uniform across reviewer types — or concentrated in a specific segment.

In [3]:
tier_order = ["Casual", "Regular", "Power", "Elite"]

seg = (
    sbux.groupby("user_activity_tier")
    .agg(
        Reviews       =("review_id",      "count"),
        Avg_Stars     =("review_stars",   "mean"),
        Avg_Sentiment =("sentiment_score","mean"),
        Pct_Critical  =("star_tier",      lambda x: round((x == "Critical").mean() * 100, 1)),
        Pct_Positive  =("star_tier",      lambda x: round((x == "Positive").mean() * 100, 1)),
    )
    .reindex(tier_order)
    .round({"Avg_Stars": 2, "Avg_Sentiment": 2})
    .reset_index()
)
seg.columns = ["Tier", "Reviews", "Avg Stars", "Avg Sentiment", "% Critical", "% Positive"]
print(seg.to_string(index=False))

   Tier  Reviews  Avg Stars  Avg Sentiment  % Critical  % Positive
 Casual     3778       2.43           0.10        64.3        30.5
Regular     3385       2.69           0.19        55.8        36.3
  Power     2468       3.32           0.43        34.0        53.6
  Elite     2044       3.60           0.56        19.6        60.9


## 3. Avg star rating & % critical by reviewer tier

In [4]:

TIER_COLORS = ["#d62728", "#f4a261", "#6aaed6", "#00704A"]  # Casual→Elite

fig = go.Figure(go.Bar(
    x=seg["Tier"],
    y=seg["Avg Stars"],
    marker_color=TIER_COLORS,
    text=[f'{v:.2f}★  ({c}% critical)' for v, c in zip(seg["Avg Stars"], seg["% Critical"])],
    textposition="outside",
))
fig.add_hline(
    y=sbux["review_stars"].mean(),
    line_dash="dot", line_color="#888888",
    annotation_text=f'Starbucks avg {sbux["review_stars"].mean():.2f}',
    annotation_position="top right",
)
fig.update_layout(
    title=dict(text="Starbucks — Avg Star Rating by Reviewer Activity Tier", font=dict(size=16)),
    xaxis_title="Reviewer Tier",
    yaxis_title="Avg Star Rating",
    plot_bgcolor="white", paper_bgcolor="white",
    yaxis=dict(showgrid=True, gridcolor="#eeeeee", range=[0, 4.5]),
    xaxis=dict(showgrid=False),
    width=700, height=420,
    margin=dict(t=60, b=50, l=60, r=40),
)
fig.write_html(str(FIGURES_DIR / "08_reviewer_tier.html"))
fig.show()


## 4. Review length vs sentiment tier

Review length is a proxy for emotional investment. Customers who feel strongly — positively or negatively — tend to write more. Examining average length across sentiment tiers reveals whether dissatisfied customers invest more effort in articulating their experience.

In [5]:

tier_sent_order = ["Critical", "Neutral", "Positive"]
len_seg = (
    sbux.groupby("star_tier")
    .agg(
        Reviews    =("review_id",     "count"),
        Avg_Length =("review_length", "mean"),
        Avg_Stars  =("review_stars",  "mean"),
    )
    .reindex(tier_sent_order)
    .round(1)
    .reset_index()
)
len_seg.columns = ["Sentiment Tier", "Reviews", "Avg Length (words)", "Avg Stars"]
print(len_seg.to_string(index=False))


Sentiment Tier  Reviews  Avg Length (words)  Avg Stars
      Critical     5557                86.4        1.3
       Neutral     1171                91.1        3.0
      Positive     4947                71.1        4.7


## Key findings

**Negative signal is concentrated in low-engagement reviewers.**
Casual reviewers — those with the fewest reviews on Yelp — account for 3,778 Starbucks reviews with an average of 2.43 stars and a 64.3% critical rate. Elite reviewers, who have the broadest range of comparison venues, average 3.60 stars with only a 19.6% critical rate. The gap between these two segments is 1.17 stars — larger than the gap between Starbucks and the overall coffee market benchmark.

**Experience level shifts perception.**
Reviewers with more dining and café experience apply a broader reference frame. A 3-star Starbucks visit rated against hundreds of venues reads differently than the same visit rated by someone with limited comparison points. This does not mean negative reviews are invalid — it means the brand's performance looks more severe to first-time or infrequent reviewers, who are likely also the most price-sensitive segment.

**Dissatisfied customers write more.**
Critical reviews average 86 words versus 71 for positive reviews — a 21% difference. Customers who have a complaint invest more effort in articulating it. This has a practical implication: negative reviews carry more textual signal for qualitative analysis, and their language is more specific and actionable than the language of positive reviews.

---

**Next: Notebook 09 — Brand Benchmarking Scorecard**